# Lab 07.2 — Discovery e Ciclo de Vida

Os records foram criados e **APPROVED** no 07.1. Aqui vemos os dois usos do
registry: **discovery** (descobrir agentes e seus metadados de risco) e
**lifecycle** (deprecating an agent that leaves production).

Status válidos: `DRAFT → PENDING_APPROVAL → APPROVED → DEPRECATED`.

## Setup

In [ ]:
import sys, json
sys.path.insert(0, "..")
from shared.utils.config import load_config, get_region
from utils import list_registry_records
import boto3

cfg = load_config()
region = get_region()
registry_id = cfg["AGENTCORE_REGISTRY_ID"]
client = boto3.client("bedrock-agentcore-control", region_name=region)

## Step 1: Discovery — listar agentes APPROVED e seus metadados de risco

In [ ]:
approved = list_registry_records(registry_id, status="APPROVED", region=region)
print(f"{len(approved)} agentes APPROVED:\n")
for r in approved:
    rid = r.get("recordId") or r.get("recordArn", "").split("/")[-1]
    detail = client.get_registry_record(registryId=registry_id, recordId=rid)
    content = (detail.get("descriptors", {}).get("custom", {}) or {}).get("inlineContent", "{}")
    meta = json.loads(content) if content else {}
    print(f"  • {r.get('name'):24s} risk={meta.get('risk_level','?'):6s} team={meta.get('team','?')}")

## Step 2: Lifecycle — deprecate an agent (APPROVED → DEPRECATED)

When an agent leaves production, `DEPRECATED` preserves the history for
auditoria sem mantê-lo como aprovado.

In [ ]:
target = next((r for r in approved if r.get("name") == "ContractAgent"), None)
if target:
    rid = target.get("recordId") or target.get("recordArn", "").split("/")[-1]
    client.update_registry_record_status(
        registryId=registry_id,
        recordId=rid,
        status="DEPRECATED",
        statusReason="Agente descontinuado — consolidado no fluxo de contratos.",
    )
    print(f"✓ ContractAgent → DEPRECATED")

## Step 3: Confirmar separação por status

In [ ]:
for st in ("APPROVED", "DEPRECATED"):
    recs = list_registry_records(registry_id, status=st, region=region)
    print(f"{st}: {[r.get('name') for r in recs]}")

## 🎓 What you learned

- **Discovery**: descobrir agentes e ler metadados de risco do descriptor CUSTOM
- **Lifecycle**: `DEPRECATED` preserva histórico (auditoria) sem manter aprovado

## Cleanup

```python
from utils import cleanup_registry
cleanup_registry(cfg["AGENTCORE_REGISTRY_ID"], region=region)
```

## Next

➡️ [Lab 08 — AgentCore Observability](../08-AgentCore-Observability/)